# 사전학습: 재료 데이터를 읽고 조회하기
AI for Materials Science — Hands-on session 1

수업에서 바로 사용할 NumPy, pandas, pymatgen과 API 조회를 연습하겠습니다.

| 절 | 할 일 | 학습 시간 |
|---|---|---:|
| 0 | 준비: 설치와 자료 내려받기 | 3분 |
| 1 | NumPy 배열과 조건 선택 | 6분 |
| 2 | CSV 읽기, 결측값 확인, 후보 선택 | 9분 |
| 3 | 화학식과 결정 구조, CIF 읽기·쓰기 | 8분 |
| 4 | 키 없이 OQMD 조회 | 5분 |
| 5 | (선택) MP에서 LiFePO₄ 조회 | +3분 |

위에서부터 한 셀씩 실행해주세요. 코드의 `##` 줄은 설명용 주석이라 실행되지 않습니다.

## 0. 준비

### 0-1. 라이브러리 설치
`pymatgen`을 설치하면 NumPy, pandas, requests도 의존 패키지로 함께 들어옵니다.

In [ ]:
!pip install -q pymatgen

### 0-2. 수업 자료 내려받기
실습 데이터(CSV, CIF, OQMD 응답 사본)는 수업 저장소의 `Data/` 폴더에 있습니다.

In [ ]:
!git clone https://github.com/kwongibaek/MS49900-AI4M.git

### 0-3. 라이브러리 불러오기
`import`는 다른 곳에서 만든 도구를 현재 노트북으로 가져오는 명령입니다.
`as`는 짧은 별명을 붙입니다.

In [ ]:
import json
import os
from pathlib import Path

import numpy as np
import pandas as pd
import requests
from IPython.display import display

### 0-4. 파일 경로 정하기
경로를 변수에 담아두면 뒤에서는 CSV와 CIF의 이름만 기억하면 됩니다.
개인 컴퓨터의 폴더 주소를 직접 적을 필요는 없습니다.

In [ ]:
## clone한 폴더, 저장소 최상위, Hand-on-session1 폴더 중 Data가 있는 곳을 찾습니다.
for candidate in [Path("MS49900-AI4M/Data"), Path("Data"), Path("../Data")]:
    if candidate.is_dir():
        DATA_DIR = candidate
        break

STEEL_CSV = DATA_DIR / "steel_strength.csv"
LFP_CIF = DATA_DIR / "LiFePO4.cif"
OQMD_CACHE = DATA_DIR / "oqmd_li_fe_p_o_contains_sample.json"

## 실습 결과는 현재 작업 폴더의 outputs/01_preclass/에 저장됩니다.
OUTPUT_DIR = Path("outputs/01_preclass")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(DATA_DIR)
sorted(path.name for path in DATA_DIR.iterdir())

## 1. NumPy: 배열에서 필요한 값 고르기 · 6분

항복강도 다섯 값을 배열에 담아보겠습니다. 배열은 여러 숫자를 한꺼번에 계산할 때 사용합니다.
여기서는 연습용 값을 쓰고, 다음 절에서 실제 실험 자료를 읽겠습니다.

### 1-1. 배열 만들기

In [ ]:
## np는 NumPy의 짧은 이름입니다. =로 오른쪽 배열을 왼쪽 변수에 저장합니다.
strength = np.array([620.0, 780.0, 910.0, 1050.0, 1180.0])

print("배열:", strength)
print("자료형과 모양:", strength.dtype, strength.shape)

### 1-2. 위치로 값 꺼내기

In [ ]:
## 위치는 0부터 셉니다. [1:4]는 위치 1, 2, 3을 고르며 끝 위치 4는 제외합니다.
print("첫 값:", strength[0])
print("마지막 값:", strength[-1])
print("일부 값:", strength[1:4])

### 1-3. 배열 전체에 같은 계산 적용하기

In [ ]:
## 배열에 숫자 하나를 더하면 모든 값에 같은 숫자가 더해집니다.
adjusted = strength + 25.0
adjusted

### 1-4. 조건으로 값 고르기
비교 연산의 결과는 값 하나가 아니라 **True/False 배열**입니다.

In [ ]:
mask = strength >= 900.0
mask

In [ ]:
## True/False 배열을 []에 넣으면 True인 위치의 값만 남습니다.
selected = strength[mask]

print("선택된 값:", selected)
print("선택된 개수와 평균:", selected.size, selected.mean())

## 2. pandas: CSV를 읽고 강재 후보 고르기 · 9분

`steel_strength.csv`에는 강재의 조성과 실험 물성이 들어 있습니다.
원소 함량은 wt%, 항복강도·인장강도는 MPa, 연신율은 %입니다.
`pd.read_csv()`로 읽은 표를 **DataFrame**이라고 부릅니다.

### 2-1. CSV 읽기

In [ ]:
steel = pd.read_csv(STEEL_CSV)
steel.head()

In [ ]:
## shape는 (행 수, 열 수), len(DataFrame)은 행 수를 반환합니다.
print("행과 열:", steel.shape)
print("전체 행 수:", len(steel))

### 2-2. 필요한 열만 고르기

In [ ]:
## 여러 열을 고를 때는 열 이름의 리스트를 [] 안에 넣습니다.
property_columns = ["yield strength", "tensile strength", "elongation"]
properties = steel[property_columns]
properties.head()

### 2-3. 결측값 확인하기
`count()`는 값이 있는 칸을, `isna().sum()`은 비어 있는 칸을 열별로 셉니다.

In [ ]:
pd.DataFrame({
    "값이 있는 개수": properties.count(),
    "결측값 개수": properties.isna().sum(),
})

### 2-4. 조건 두 개 만들기
항복강도 1500 MPa 이상, 연신율 10% 이상인 강재를 찾아보겠습니다.

In [ ]:
## notna()는 값이 있는지 확인합니다. 결측값을 조건에서 먼저 걸러냅니다.
strength_condition = steel["yield strength"] >= 1500.0
elongation_condition = steel["elongation"].notna() & (steel["elongation"] >= 10.0)

print("강도 조건 통과:", int(strength_condition.sum()))
print("연신율 조건 통과:", int(elongation_condition.sum()))

### 2-5. 두 조건을 모두 만족하는 행 고르기

In [ ]:
## &는 두 조건을 모두 만족해야 한다는 뜻입니다. loc[조건]으로 행을 고릅니다.
candidates = steel.loc[strength_condition & elongation_condition].copy()
candidates = candidates.sort_values("yield strength", ascending=False)

print("두 조건 모두 통과:", len(candidates))
display(candidates[["formula", *property_columns]].head())

### 2-6. 결과 저장하기

In [ ]:
## index=False는 DataFrame의 행 번호를 CSV의 별도 열로 저장하지 않는 설정입니다.
candidate_csv = OUTPUT_DIR / "steel_candidates.csv"
candidates.to_csv(candidate_csv, index=False)

print("저장한 파일:", candidate_csv)

## 3. pymatgen: 화학식과 결정 구조 다루기 · 8분

`Composition`은 원소의 종류와 개수를, `Structure`는 격자와 원자 위치까지 다룹니다.
먼저 LiFePO₄ 화학식을 읽고, 간단한 Si 구조를 만들어 CIF로 저장한 뒤, 준비된 LiFePO₄ CIF를 읽겠습니다.

실험 결정 구조를 찾을 때 ICSD는 일반적으로 이용 라이선스가 필요하고, COD는 공개 접근이 가능합니다.
여기서 읽는 CIF는 pymatgen에 공개된 예제 파일입니다.

### 3-1. 화학식 해석하기

In [ ]:
from pymatgen.core import Composition, Lattice, Structure

## Composition에 화학식을 넣으면 원소 종류와 개수를 해석해줍니다.
lfp_composition = Composition("LiFePO4")

print("간단한 화학식:", lfp_composition.reduced_formula)
print("화학식의 원자 수 합:", lfp_composition.num_atoms)

In [ ]:
## 원자 분율은 해당 원소의 원자 수를 전체 원자 수로 나눈 값입니다.
lfp_composition.fractional_composition.as_dict()

### 3-2. 대칭으로 구조 만들기

In [ ]:
## 공간군 대칭을 적용해 Si 원자들의 위치를 만듭니다. 격자 길이의 단위는 Å입니다.
silicon = Structure.from_spacegroup(
    "Fd-3m", Lattice.cubic(5.431), species=["Si"], coords=[[0, 0, 0]],
)

print("격자 길이:", silicon.lattice.abc)
print("이 셀의 원자 자리 수:", len(silicon))

### 3-3. CIF 파일로 저장하기

In [ ]:
## 구조 객체를 CIF 파일로 저장하면 다른 프로그램에서도 읽을 수 있습니다.
silicon_cif = OUTPUT_DIR / "silicon.cif"
silicon.to(filename=str(silicon_cif), fmt="cif")

print("저장한 구조:", silicon_cif)

### 3-4. CIF 파일 읽기

In [ ]:
## from_file()은 CIF를 읽어 Structure 객체로 바꿉니다.
lfp_structure = Structure.from_file(LFP_CIF)

print("화학식:", lfp_structure.composition.reduced_formula)
print("격자 길이:", lfp_structure.lattice.abc)
print("원자 자리 수:", len(lfp_structure))

In [ ]:
## [0]으로 첫 원자 자리를 꺼냅니다. 분율 좌표는 격자 벡터를 기준으로 한 좌표입니다.
print("첫 원소:", lfp_structure[0].species_string)
print("첫 원자의 분율 좌표:", lfp_structure[0].frac_coords)

## 4. OQMD: API 키 없이 계산 자료 조회하기 · 5분

API는 주소와 검색 조건을 보내면 프로그램이 읽을 수 있는 자료를 돌려줍니다.
OQMD에서 **Li, Fe, P, O를 모두 포함하는 자료를 최대 20건** 받아보겠습니다.

요청 → 응답 상태 확인 → JSON 읽기 → DataFrame 변환 순서로 진행합니다.
`delta_e`는 원자당 형성 에너지, `stability`는 convex hull 위 거리이며 둘 다 eV/atom 단위의 계산값입니다.
OQMD와 MP의 에너지는 기준 에너지·구조·보정 방법을 맞추기 전에 행별로 직접 비교하면 안 됩니다.

### 4-1. 검색 조건 만들기
`element_set=(Li,Fe,P,O)`는 추가 원소도 허용합니다. 정확히 네 원소만 원하면 `AND ntypes=4`를 더합니다.

In [ ]:
## fields는 응답에서 가져올 항목이고, filter는 검색 조건입니다.
OQMD_URL = "https://oqmd.org/oqmdapi/formationenergy"
OQMD_FIELDS = ["name", "entry_id", "spacegroup", "ntypes", "natoms",
               "delta_e", "stability", "band_gap", "prototype", "icsd_id"]
OQMD_PARAMS = {
    "fields": ",".join(OQMD_FIELDS),
    "limit": 20,
    "offset": 0,
    "format": "json",
    "filter": "element_set=(Li,Fe,P,O)",
}

OQMD_PARAMS

### 4-2. 요청 보내기
서버가 느리면 같은 조건으로 미리 받아 둔 응답 사본으로 전환됩니다.
정상적인 처리이므로 표시된 출처를 확인하고 계속 진행하세요.

In [ ]:
try:
    ## 실제 HTTP 요청을 보내고, 오류가 없으면 JSON을 Python 자료로 읽습니다.
    response = requests.get(OQMD_URL, params=OQMD_PARAMS, timeout=30)
    response.raise_for_status()
    oqmd_payload = response.json()
    oqmd_source = "OQMD 실시간 HTTP 응답"
except requests.RequestException as error:
    ## 요청이 실패하면 저장소에 함께 담긴 같은 조건의 응답 사본을 사용합니다.
    oqmd_payload = json.loads(OQMD_CACHE.read_text(encoding="utf-8"))
    oqmd_source = f"OQMD 캐시 ({type(error).__name__})"

print("데이터 출처:", oqmd_source)
print("응답 상태:", oqmd_payload["response_message"])

### 4-3. 응답을 표로 바꾸기
전체 검색 건수와 현재 페이지의 행 수는 다를 수 있습니다.

In [ ]:
oqmd_table = pd.DataFrame(oqmd_payload["data"], columns=OQMD_FIELDS)

print("현재 페이지 행 수:", len(oqmd_table))
print("전체 검색 건수:", oqmd_payload["meta"]["data_available"])
display(oqmd_table[["name", "entry_id", "ntypes", "delta_e", "stability"]])

## 5. 선택: MP에서 LiFePO₄ 한 번 조회하기 · 3분

여기까지가 기본 사전학습입니다. MP API 키가 있는 학생은 같은 요청 방식을 한 번 더 써보세요.
키가 없으면 이 절은 건너뛰어도 됩니다.

MP에서도 주소와 조건으로 자료를 요청할 수 있습니다. 같은 화학식에 구조가 여러 개라면 여러 행이 나옵니다.
이번에는 최대 10행만 확인하고, 자세한 검색 조건과 `MPRester` 사용법은 02 노트북에서 다루겠습니다.

### 5-1. API 키 입력하기
키는 화면에 표시되지 않습니다. 키를 코드나 제출 파일에 직접 적지 마세요.

In [ ]:
from getpass import getpass

## 환경변수에 키가 있으면 그것을 쓰고, 없으면 숨김 입력으로 받습니다.
MP_API_KEY = os.getenv("MP_API_KEY", "").strip()
if not MP_API_KEY:
    MP_API_KEY = getpass("MP API key: ").strip()

### 5-2. 조회하고 표로 보기

In [ ]:
if not MP_API_KEY:
    raise ValueError("MP API 키가 필요합니다. 이 절은 선택 사항이므로 건너뛰어도 됩니다.")

MP_FIELDS = ["material_id", "formula_pretty", "band_gap", "energy_above_hull"]

## OQMD와 달리 MP에는 인증 키를 headers로 함께 보냅니다.
mp_response = requests.get(
    "https://api.materialsproject.org/materials/summary/",
    headers={"X-API-KEY": MP_API_KEY},
    params={"formula": "LiFePO4", "_fields": ",".join(MP_FIELDS), "_limit": 10},
    timeout=30,
)
mp_response.raise_for_status()

## 응답 목록을 표로 바꾸어 밴드갭(eV)과 hull 위 에너지(eV/atom)를 확인합니다.
mp_table = pd.DataFrame(mp_response.json()["data"], columns=MP_FIELDS)
print("이번 요청에서 받은 행 수:", len(mp_table))
display(mp_table)